In [1]:
!pip install mlflow dagshub optuna

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.5/263.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.1/197.1 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.

In [4]:
import numpy as np
import pandas as pd
import utils_data_clean_preprocessed
import mlflow
import optuna
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer,TransformedTargetRegressor
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder, MinMaxScaler, PowerTransformer, OrdinalEncoder
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor,StackingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [5]:
import dagshub
dagshub.init(repo_owner='Iamkartikey44', repo_name='Food_Delivery_Time_Prediction', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=e57256f3-96c3-4a76-b590-9e211975f661&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=5c4fbb1e3882fe108eaeb546a6d074268aca7809bd6166f8594d6ab4eccfd63f




Accessing as Iamkartikey44

Initialized MLflow to track repo "Iamkartikey44/Food_Delivery_Time_Prediction"

Repository Iamkartikey44/Food_Delivery_Time_Prediction initialized!

In [6]:
#Mlflow Experiment
mlflow.set_experiment("Exp3 - Stacking Regressor")

<Experiment: artifact_location='mlflow-artifacts:/4529f492c53e494cac2d43155358eb44', creation_time=1772964048951, experiment_id='2', last_update_time=1772964048951, lifecycle_stage='active', name='Exp3 - Stacking Regressor', tags={}, workspace='default'>

In [7]:
from sklearn import set_config
set_config(transform_output='pandas')

In [8]:
#Load the data
df = pd.read_csv("swiggy.csv")

In [9]:
#Load the clean data
df1 = utils_data_clean_preprocessed.perform_data_cleaning(df)

In [10]:
df1.head()

,age,ratings,weather,traffic,vehicle_condition,type_of_order,type_of_vehicle,multiple_deliveries,festival,city_type,time_taken,is_weekend,pickup_time_minutes,order_time_of_day,distance,distance_type
0,37.0,4.9,sunny,high,2,snack,motorcycle,0.0,no,urban,24,1,15.0,morning,0.0,short
1,34.0,4.5,stormy,jam,2,snack,scooter,1.0,no,metropolitian,33,0,5.0,evening,0.0,short
2,23.0,4.4,sandstorms,low,0,drinks,motorcycle,1.0,no,urban,26,1,15.0,morning,0.0,short
3,38.0,4.7,sunny,medium,0,buffet,motorcycle,1.0,no,metropolitian,21,0,10.0,evening,0.0,short
4,32.0,4.6,cloudy,high,1,snack,scooter,1.0,no,metropolitian,30,1,15.0,afternoon,0.0,short


In [11]:
df1.columns

Index(['age', 'ratings', 'weather', 'traffic', 'vehicle_condition',
       'type_of_order', 'type_of_vehicle', 'multiple_deliveries', 'festival',
       'city_type', 'time_taken', 'is_weekend', 'pickup_time_minutes',
       'order_time_of_day', 'distance', 'distance_type'],
      dtype='object')

In [12]:
df1.isna().sum()

,0
age,0
ratings,0
weather,0
traffic,0
vehicle_condition,0
type_of_order,0
type_of_vehicle,0
multiple_deliveries,0
festival,0
city_type,0


In [13]:
df1.duplicated().sum()

np.int64(24)

In [14]:
temp_df = df1.copy()

In [15]:
# split into X and y

X = temp_df.drop(columns='time_taken')
y = temp_df['time_taken']

In [16]:
#Train Test Split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [17]:
print(f"The size of train data is: {X_train.shape}")
print(f"The shape of test data is: {X_test.shape}")

The size of train data is: (30156, 15)
The shape of test data is: (7539, 15)


In [18]:
# missing values in train data
X_train.isna().sum()

,0
age,0
ratings,0
weather,0
traffic,0
vehicle_condition,0
type_of_order,0
type_of_vehicle,0
multiple_deliveries,0
festival,0
city_type,0


In [19]:
#Transform target column
pt = PowerTransformer()

y_train_pt = pt.fit_transform(y_train.values.reshape(-1,1))
y_test_pt = pt.transform(y_test.values.reshape(-1,1))

## Pre Processing Pipeline

In [20]:
num_cols = ["age","ratings","pickup_time_minutes","distance"]

nominal_cat_cols = ['weather',
                    'type_of_order',
                    'type_of_vehicle',
                    "festival",
                    "city_type",
                    "is_weekend",
                    "order_time_of_day"]

ordinal_cat_cols = ["traffic","distance_type"]

In [21]:
#Generate order for ordinal encoding
traffic_order = ['low','medium','high','jam']
distance_type_order = ['short','medium','long','very_long']

In [22]:
# unique categories the ordinal columns
for col in ordinal_cat_cols:
    print(col,X_train[col].unique())

traffic ['jam' 'medium' 'high' 'low']
distance_type ['short']
Categories (4, object): ['short' < 'medium' < 'long' < 'very_long']


In [23]:
# build a preprocessor
preprocessor = ColumnTransformer(transformers=[
    ("scale",MinMaxScaler(),num_cols),
    ("nominal_encode",OneHotEncoder(handle_unknown="ignore",drop='first',sparse_output=False),nominal_cat_cols),
    ('ordinal_encode',OrdinalEncoder(categories=[traffic_order,distance_type_order],encoded_missing_value=-999,handle_unknown='use_encoded_value',unknown_value=-1),ordinal_cat_cols)
],remainder='passthrough',n_jobs=-1,force_int_remainder_cols=False,verbose_feature_names_out=False)

In [24]:
preprocessor

ColumnTransformer(force_int_remainder_cols=False, n_jobs=-1,
                  remainder='passthrough',
                  transformers=[('scale', MinMaxScaler(),
                                 ['age', 'ratings', 'pickup_time_minutes',
                                  'distance']),
                                ('nominal_encode',
                                 OneHotEncoder(drop='first',
                                               handle_unknown='ignore',
                                               sparse_output=False),
                                 ['weather', 'type_of_order', 'type_of_vehicle',
                                  'festival', 'city_type', 'is_weekend',
                                  'order_time_of_day']),
                                ('ordinal_encode',
                                 OrdinalEncoder(categories=[['low', 'medium',
                                                             'high', 'jam'],
                                                            ['short', 'medium',
                                                             'long',
                                                             'very_long']],
                                                encoded_missing_value=-999,
                                                handle_unknown='use_encoded_value',
                                                unknown_value=-1),
                                 ['traffic', 'distance_type'])],
                  verbose_feature_names_out=False)

In [25]:
# build the pipeline
processing_pipeline  = Pipeline(steps=[
    ("preprocess",preprocessor)
])

In [ ]:
processing_pipeline

Pipeline(steps=[('preprocess',
                 ColumnTransformer(force_int_remainder_cols=False, n_jobs=-1,
                                   remainder='passthrough',
                                   transformers=[('scale', MinMaxScaler(),
                                                  ['age', 'ratings',
                                                   'pickup_time_minutes',
                                                   'distance']),
                                                 ('nominal_encode',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['weather', 'type_of_order',
                                                   'type_of_vehicle',
                                                   'festival', 'city_type',
                                                   'is_weekend',
                                                   'order_time_of_day']),
                                                 ('ordinal_encode',
                                                  OrdinalEncoder(categories=[['low',
                                                                              'medium',
                                                                              'high',
                                                                              'jam'],
                                                                             ['short',
                                                                              'medium',
                                                                              'long',
                                                                              'very_long']],
                                                                 encoded_missing_value=-999,
                                                                 handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['traffic',
                                                   'distance_type'])],
                                   verbose_feature_names_out=False))])

In [26]:
#Data Preprocessing
X_train_trans = processing_pipeline.fit_transform(X_train)
X_test_trans = processing_pipeline.transform(X_test)

In [47]:
xgb_params = {'n_estimators': 415,
 'max_depth': 20,
 'learning_rate': 0.29263456258035475,
 'subsample': 0.8759669065136347,
 'colsample_bytree': 0.7213914971809194,
 'min_child_weight': 17,
 'gamma': 1.0059958323704052,
 'reg_lambda': 37.94112216359326,
 'reg_alpha': 0.5695261402488025}

lgbm_params ={'n_estimators': 398,
 'max_depth': 30,
 'learning_rate': 0.17723976615372525,
 'subsample': 0.6668154532640596,
 'min_child_weight': 13,
 'min_split_gain': 0.00042029710843746043,
 'reg_lambda': 75.85847527149905}

rf_params =  {'n_estimators': 500,
 'max_depth': 14,
 'max_features': None,
 'min_samples_split': 9,
 'min_samples_leaf': 6,
 'max_samples': 0.9797814656591307}




In [48]:
xgb_reg = XGBRegressor(**xgb_params)
lgbm_reg = LGBMRegressor(**lgbm_params)
rf_reg = RandomForestRegressor(**rf_params)

In [ ]:
# build the best models
best_rf_params = {'n_estimators': 479,
 'criterion': 'squared_error',
 'max_depth': 17,
 'max_features': None,
 'min_samples_split': 9,
 'min_samples_leaf': 2,
 'max_samples': 0.6603673526197067}

best_lgbm_params = {'n_estimators': 154,
 'max_depth': 27,
 'learning_rate': 0.22234435854395157,
 'subsample': 0.7592213724048168,
 'min_child_weight': 20,
 'min_split_gain': 0.004604680609280751,
 'reg_lambda': 97.81002379097947}

best_rf = RandomForestRegressor(**best_rf_params)
best_lgbm = LGBMRegressor(**best_lgbm_params)

In [49]:
def objective(trial):
  with mlflow.start_run(nested=True):
    meta_model_name = trial.suggest_categorical("model",["LR","KNN","DT"])

    if meta_model_name =='LR':
      meta = LinearRegression()

    elif meta_model_name =='KNN':
      n_neighbors = trial.suggest_int("n_neighbors",1,50)
      weights_knn = trial.suggest_categorical("weights_knn",["uniform","distance"])
      meta = KNeighborsRegressor(n_neighbors=n_neighbors,weights=weights_knn,n_jobs=-1)

    elif meta_model_name =='DT':
      max_depth_dt = trial.suggest_int("max_depth_dt",1,10)
      min_sample_split_dt = trial.suggest_int("min_samples_split_dt",2,10)
      min_samples_leaf_dt = trial.suggest_int("min_samples_leaf_dt",1,10)
      meta = DecisionTreeRegressor(max_depth=max_depth_dt,min_samples_split=min_sample_split_dt,min_samples_leaf=min_samples_leaf_dt,random_state=42)

    #Log meta model name
    mlflow.log_param("meta_model_name",meta_model_name)


    #Stacking regressor
    #stacking_reg = StackingRegressor(estimators=[('rf',best_rf),('lgbm',best_lgbm)],final_estimator=meta,cv=5,n_jobs=-1)
    stacking_reg = StackingRegressor(estimators=[('XGBoost',xgb_reg),('lgbm',lgbm_reg),('rf',rf_reg)],final_estimator=meta,cv=5,n_jobs=-1)

    #Build Transformed regressor
    model = TransformedTargetRegressor(regressor=stacking_reg,transformer=pt)

    #Train the model
    model.fit(X_train_trans,y_train)

    #Predict on test data
    y_pred_test = model.predict(X_test_trans)

    #Mean Absoulte error
    error = mean_absolute_error(y_test,y_pred_test)

    #Log Error
    mlflow.log_metric("MAE",error)

    return error

In [50]:
# create optuna study
study = optuna.create_study(direction='minimize')

with mlflow.start_run(run_name='best_model'):
  study.optimize(objective,n_trials=10,n_jobs=-1,show_progress_bar=True)
  mlflow.log_params(study.best_params)
  mlflow.log_metric("Best Score",study.best_value)

[I 2026-03-08 18:12:39,658] A new study created in memory with name: no-name-2d31d338-c9e7-4bb7-afa6-49460a563db6


  0%|          | 0/10 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning:

A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.



🏃 View run trusting-lynx-617 at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/2/runs/aca88dc712414d0ea1338f88ac499442
🧪 View experiment at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/2
[I 2026-03-08 18:16:54,805] Trial 0 finished with value: 3.709892764117285 and parameters: {'model': 'LR'}. Best is trial 0 with value: 3.709892764117285.
🏃 View run calm-hog-352 at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/2/runs/f6b77ec6fc2646a8a023f1a29a543bb2
🧪 View experiment at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/2
[I 2026-03-08 18:17:13,655] Trial 1 finished with value: 3.744727575153803 and parameters: {'model': 'DT', 'max_depth_dt': 4, 'min_samples_split_dt': 7, 'min_samples_leaf_dt': 5}. Best is trial 0 with value: 3.709892764117285.


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning:

A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.



...

🏃 View run charming-calf-349 at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/2/runs/41a3ec0c87354caa9a7b5078e8aeb62b
🧪 View experiment at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/2
[I 2026-03-08 18:29:59,404] Trial 6 finished with value: 3.709899482669589 and parameters: {'model': 'LR'}. Best is trial 5 with value: 3.709082919049857.


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning:

A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.



🏃 View run masked-shoat-705 at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/2/runs/d772382e48614b95a2893bc9e189dc43
🧪 View experiment at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/2
[I 2026-03-08 18:30:16,994] Trial 7 finished with value: 3.8447189074946797 and parameters: {'model': 'DT', 'max_depth_dt': 3, 'min_samples_split_dt': 2, 'min_samples_leaf_dt': 6}. Best is trial 5 with value: 3.709082919049857.


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning:

A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.



🏃 View run bold-cub-446 at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/2/runs/0361fa2609b842ae81919ee77b2b8afc
🧪 View experiment at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/2
[I 2026-03-08 18:34:19,646] Trial 8 finished with value: 3.7091299555032022 and parameters: {'model': 'LR'}. Best is trial 5 with value: 3.709082919049857.
🏃 View run learned-fish-680 at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/2/runs/9b0a8d0be6644fe0846ed594f5be5bf1
🧪 View experiment at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/2
[I 2026-03-08 18:34:31,880] Trial 9 finished with value: 5.20285833244446 and parameters: {'model': 'DT', 'max_depth_dt': 1, 'min_samples_split_dt': 5, 'min_samples_leaf_dt': 2}. Best is trial 5 with value: 3.709082919049857.
🏃 View run best_model at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.

In [45]:
# best parameter value
best_params = study.best_params
best_params

{'model': 'LR'}

In [44]:
# parameter value counts
study.trials_dataframe()["params_model"].value_counts()

,count
params_model,
DT,5
KNN,3
LR,2


In [46]:
# mean scores for each meta estimator type
study.trials_dataframe().groupby(by="params_model")['value'].mean().sort_values()

,value
params_model,
LR,3.710389
DT,3.861742
KNN,3.865330


In [ ]:
# mean scores for each meta estimator type
study.trials_dataframe().groupby(by="params_model")['value'].mean().sort_values()

,value
params_model,
LR,3.721409
KNN,3.950509
DT,3.999897


In [51]:
study.best_value

3.709082919049857

In [41]:
study.best_value #3M

3.710187360655401

In [32]:
# best score
study.best_value

3.73268421566814

In [ ]:
# best score
study.best_value

3.7194792641423384

In [53]:
# optimization history plot
optuna.visualization.plot_optimization_history(study)

In [54]:
# parallel coord plot
optuna.visualization.plot_parallel_coordinate(study,params=["model"])